# Part 1: Data Loading & Setup

## Q1. Spark Session & Data Loading


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**Create a SparkSession**

In [2]:
spark = SparkSession.builder.appName("Module9").master("local[*]").getOrCreate()

**Load sales_data.csv**

In [3]:
sales_df = spark.read\
        .option("header",True)\
        .option("inderSchema",True)\
        .csv(r"C:\Users\aman.rajput\Downloads\Module_9_Assignment\sales_data.csv")

**Infer schema and display schema**

In [4]:
print(sales_df.printSchema())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_timestamp: string (nullable = true)

None


## Q2. Data Type Conversion

**Convert order_date to DateType**<br>
**Convert order_timestamp to TimestampType**

In [5]:
sdf = sales_df.withColumn('order_date',to_date(col("order_date")))\
        .withColumn('order_timestamp',to_timestamp("order_timestamp"))\
        .withColumn('order_amount',col('order_amount').cast("int"))

In [6]:
sdf.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



# Part 2: Aggregate Functions


## Q3. Overall Sales Metrics


**Total sales amount (sum)**

In [7]:
sdf.agg(sum('order_amount').alias("total_sales_amount")).show()

+------------------+
|total_sales_amount|
+------------------+
|            359000|
+------------------+



**Average order amount (avg)**

In [8]:
sdf.agg(avg('order_amount').alias('Average_order_amount')).show()

+--------------------+
|Average_order_amount|
+--------------------+
|             44875.0|
+--------------------+



**Maximum and minimum order amount**

In [9]:
sdf.agg(max('order_amount').alias('Maximum_order_amount')).show()

+--------------------+
|Maximum_order_amount|
+--------------------+
|               80000|
+--------------------+



In [10]:
sdf.agg(min('order_amount').alias('Minimum_order_amount')).show()

+--------------------+
|Minimum_order_amount|
+--------------------+
|               20000|
+--------------------+



## Q4. Region-wise Analysis:


For each region, calculate:

- Total sales
- Average sales
- Order count


In [11]:
sdf.groupBy('region')\
    .agg(
        sum("order_amount").alias("Total_sales"),
        avg("order_amount").alias("Avg_sales"),
        count("order_id").alias("Order_count")
        ).show()

+------+-----------+------------------+-----------+
|region|Total_sales|         Avg_sales|Order_count|
+------+-----------+------------------+-----------+
| South|     102000|           51000.0|          2|
|  East|     102000|           51000.0|          2|
|  West|      28000|           28000.0|          1|
| North|     127000|42333.333333333336|          3|
+------+-----------+------------------+-----------+



## Q5. Customer Count:


**Number of distinct customers using countDistinct()**

In [13]:
sdf.agg(
    count_distinct("customer_id").alias("Number_of_distinct_customers")
).show()

+----------------------------+
|Number_of_distinct_customers|
+----------------------------+
|                           4|
+----------------------------+



## Q6. Product-wise Aggregation:


- collect_list(order_amount)
- collect_set(order_amount)


In [14]:
sdf.agg(
    collect_list("order_amount").alias("all_order_amount"),
    collect_set("order_amount").alias("unique_order_amount")
).show()

+--------------------+--------------------+
|    all_order_amount| unique_order_amount|
+--------------------+--------------------+
|[75000, 30000, 20...|[20000, 22000, 28...|
+--------------------+--------------------+



# Part 3: Window Functions – Ranking

## Q7. Regional Ranking:

For each region,
- Assign row_number() based on highest order amount
- Assign rank() and dense_rank()


In [16]:
windowSpec = Window.partitionBy("region").orderBy(col("order_amount").desc())

In [19]:
sdf.withColumn("row_number",row_number().over(windowSpec))\
    .withColumn("rank",rank().over(windowSpec))\
    .withColumn("dense_rank",dense_rank().over(windowSpec))\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_number|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|   1|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|   2|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|   2|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|         3|   3|         3|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|   1|         1|
|    1002|       C0

## Q8. Top Orders per Region

**Find the top 2 highest orders per region using window functions.**

In [21]:
sdf.withColumn("row_number",row_number().over(windowSpec))\
    .withColumn("rank",rank().over(windowSpec))\
    .withColumn("dense_rank",dense_rank().over(windowSpec))\
    .filter(col("dense_rank") <= 2)\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_number|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|   1|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|   2|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|   2|         2|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|   1|         1|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|         2|   2|         2|
|    1005|       C0

# Part 4: Window Functions – Analytical


## Q9. Order Comparison per Customer


For each customer:
- Use lag() to show previous order amount
- Use lead() to show next order amount


In [22]:
windowSpec = Window.partitionBy("customer_id").orderBy(col("order_date"))

In [24]:
sdf.withColumn("previous_order_amount",lag("order_amount",1,0).over(windowSpec))\
    .withColumn("next_order_amount",lead("order_amount",1,0).over(windowSpec))\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+---------------------+-----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|previous_order_amount|next_order_amount|
+--------+-----------+------+-------+------------+----------+-------------------+---------------------+-----------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|                    0|            20000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|                75000|            32000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|                20000|                0|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|                    0|            72000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|                30000|                0|
|    1004|       C003|  

## Q10. Running Total

- Calculate running total (cumulative sum) of order amount:
- Partition by customer_id
- Order by order_date


In [28]:
cumulWindow = Window.partitionBy("customer_id")\
                    .orderBy("order_date")\
                    .rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [29]:
sdf.withColumn("running_total_order_amt",sum("order_amount").over(cumulWindow)).show()

+--------+-----------+------+-------+------------+----------+-------------------+-----------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|running_total_order_amt|
+--------+-----------+------+-------+------------+----------+-------------------+-----------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|                  75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|                  95000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|                 127000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|                  30000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|                 102000|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|                  80000|
|    1008|       C003|  East| Tablet|       22

# Part 5: Window Functions – Aggregates

## Q11. Average Order per Customer


- Add a column avg_customer_order showing the average order amount per customer using window functions

In [36]:
windowSpec = Window.partitionBy("customer_id")\
                    .orderBy("order_date")\
                    .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)

In [35]:
sdf.withColumn("avg_customer_order",round(avg("order_amount").over(windowSpec),2)).show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_customer_order|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|          42333.33|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|          42333.33|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|          42333.33|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           51000.0|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           51000.0|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           51000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|          

## Q12. Maximum Order per Region

- Add a column max_region_order showing the maximum order amount within each region

In [39]:
windowSpec = Window.partitionBy("region")

In [40]:
sdf.withColumn("max_region_order",max("order_amount").over(windowSpec)).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|max_region_order|
+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           80000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|           80000|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|           75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|           75000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|           75000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           72000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           72000|
|    1005|

# Part 6: Date & Timestamp Functions


## Q13. Date Components Extraction:


**Year, Month, Day from order_date**

In [42]:
sdf.withColumn("year",year("order_date"))\
    .withColumn("month",month("order_date"))\
    .withColumn("day",dayofmonth("order_date"))\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|year|month|day|
+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|2023|    1| 10|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|2023|    1| 12|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|2023|    2|  5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|2023|    2| 20|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|2023|    3|  1|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|2023|    3| 15|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|2023|    3| 18|
|    1008|       C003|  East| 

## Q14. Date Difference:


**Days between today and order_date using datediff()**

In [46]:
today = current_date().cast("date")

In [51]:
sdf.withColumn("days_between_today_and_order_date",date_diff(today,col("order_date"))).show()

+--------+-----------+------+-------+------------+----------+-------------------+---------------------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|days_between_today_and_order_date|
+--------+-----------+------+-------+------------+----------+-------------------+---------------------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|                             1172|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|                             1170|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|                             1146|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|                             1131|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|                             1122|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 

## Q15. Create new columns:


- order_year
- order_month
- order_week

In [52]:
sdf.withColumn("order_year",year("order_date"))\
    .withColumn("order_month",month("order_date"))\
    .withColumn("order_week",weekofyear("order_date"))\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      2023|          1|         2|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2023|          1|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      2023|          2|         5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      2023|          2|         8|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 

## Q16. Date-based Filtering:


**Placed in March 2023**

In [53]:
sdf.withColumn("order_year",year("order_date"))\
    .withColumn("order_month",month("order_date"))\
    .withColumn("order_week",weekofyear("order_date"))\
    .filter((col("order_month") == 3) & (col("order_year") == 2023))\
    .show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|      2023|          3|        11|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2023|          3|        11|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+



# Part 7: Real-World ETL Scenarios


## Q17. Monthly Sales Trend:


**Calculate monthly total sales (group by Year + Month)**

In [57]:
sdf.withColumn("order_year",year("order_date"))\
    .withColumn("order_month",month("order_date"))\
    .withColumn("order_week",weekofyear("order_date"))\
    .groupBy([col("order_month"),col("order_year")]).agg(
        sum("order_amount").alias("total_order_amount")
    )\
    .orderBy([col("order_month"),col("order_year")])\
    .show()

+-----------+----------+------------------+
|order_month|order_year|total_order_amount|
+-----------+----------+------------------+
|          1|      2023|            105000|
|          2|      2023|            100000|
|          3|      2023|            132000|
|          4|      2023|             22000|
+-----------+----------+------------------+



## Q18. Customer Activity Analysis:


**Identify customers who have placed more than 2 orders**

In [59]:
sdf.groupBy("customer_id").agg(count("order_id").alias("order_count")).filter(col("order_count") > 2).show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|       C001|          3|
+-----------+-----------+



In [60]:
spark.stop()